# Ablation: tiny_cnn + MNIST, FULL participation (p=1.0), 2 rounds

**What this notebook is, and is not.** This is a scalability/
correctness stress test, not a convergence experiment. It exists to
answer one reviewer question directly: *"is the protocol actually
capable of training with all 50 clients, or does it only work at the
paper's chosen `sampling_rate: 0.25`?"* -- see `ARCHITECTURE.md`
SS5.5.1 for the full framing.

**Industry framing (for the paper's industrial-application section):**
production cross-device FL systems -- Google's and Apple's deployments
over millions to billions of devices -- never require full-population
participation either; they sample a small cohort per round by
construction, for the same device-availability/bandwidth/compute-
budget reasons this paper uses `p=0.25` at MVP scale (`n=50`).
Partial participation is the industry-standard operating mode, not a
workaround unique to this paper. This notebook is the empirical half
of that argument: it proves the protocol COULD run all 50 real every
round, it just chooses not to, for the same reason production systems
choose not to.

**What changes from the main `RUN_MVP_TINY_CNN_MNIST.ipynb` run**:
only `CFG['protocol']['sampling_rate']` (forced to `1.0` here, so the
VRF self-sampling gate `b_k` passes for every one of the 50 clients --
zero dummy/zero-vector submissions this run) and the round count
(`2`, not `20`). Every other cell -- clone/install/sanity/config/
dataset/ring/model/checkpoint-setup/eval -- is IDENTICAL to the main
run notebook, imported verbatim from
`notebooks/_build_mvp_run_notebooks.py`, not reimplemented. No change
to `fl/round_loop.py`, `fl/client.py`, `fl/server.py`, or
`fl/aggregation.py` -- same pipeline, different config value.

**Why only 2 rounds, not 20.** All 50 clients running real local SGD
every round (instead of ~12 of 50) costs ~4x the GPU time of the main
matrix. This paper isn't claiming an accuracy number at `p=1.0` --
only that the pipeline doesn't structurally require subsampling to
function. Round 0 completing without a crypto/shape/FloatingPointError
and round 1 confirming it wasn't a fluke is sufficient evidence for
that claim; a third of your GPU budget is not worth spending on a
result nobody will cite.

**Checkpoint durability: git, not Drive**, same pattern as the main
matrix (FIX.md F17) -- this notebook's own dedicated
`colab-checkpoints-ablation-case1-tiny_cnn` branch, distinct from every main-matrix and
attack-matrix branch, so it can never collide with or be confused for
a real training checkpoint.


In [ ]:
# 1. Clone/pull the repo. (No Drive mount -- checkpoints go through
# git instead, see the git-checkpoint-setup cell further down and this
# notebook's own intro markdown for why.)
import os

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP


In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121


**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).


In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())


In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)


In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")


In [ ]:
# 7. Build the model.
# ~20.6K params, matches WhiteCloak/PQS-BFL's model scale (see fl/models.py build_tiny_cnn docstring).
import torch
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)

# GPU fix (2026-11-16): build_model() returns a plain nn.Module on
# CPU -- nothing downstream (fl/client.py, fl/round_loop.py) ever
# moved it or the per-batch data onto CUDA before this fix, so training
# ran entirely on CPU even on a GPU Colab runtime. Moving the model
# here, once, before the run starts is now the ONLY place that needs
# to pick the device -- fl/client.py's local_train_delta reads
# `next(model.parameters()).device` and moves each mini-batch to match,
# and fl/round_loop.py's checkpoint/resume path follows the same model
# device automatically (see both files' own 2026-11-16 comments).
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print("training device:", device)

flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


## Checkpoint durability via git (not Drive)

**Why a separate clone, a separate branch, and a callback -- not just
`git commit` inside `/content/FL-ZKP`:**
- A SEPARATE clone (`/content/FL-ZKP-checkpoints`) so committing a
  checkpoint can never touch the code checkout this notebook is actually
  importing from -- no risk of a `git checkout`/`reset` on the
  checkpoint side disturbing `sys.path`-imported modules mid-run.
- A dedicated branch (shown as `CKPT_BRANCH` in the cell below), never
  `main` -- small binary commits (one per round, re-run disconnects
  add more) have no business sitting in the same history a reviewer
  sees when they clone this repo to read the paper's actual code.
  **Each of the 4 MVP run notebooks gets its OWN branch** (FIX.md F17,
  2026-11-17) -- an earlier version shared one branch across all four,
  keyed apart only by checkpoint filename, and every single push
  collided when they ran concurrently (observed live: 12/12 rounds
  failed), even with a fetch+rebase retry (F16). A dedicated ref per
  notebook removes the race at the git level instead of mitigating it.
- `fl/round_loop.py`'s `on_checkpoint_saved` hook (2026-11-16) keeps
  the FL library itself git-agnostic -- it just calls back with
  `(checkpoint_path, round_idx)` after every local save; all the git
  mechanics below live here, in the notebook, not in the library.

**You'll need a GitHub Personal Access Token** with push access to
`YashasweeNarayan/FL-ZKP` (classic PAT, `repo` scope, or a fine-grained
token scoped to just this repo). The cell below prompts for it via
`getpass` (never echoed to the cell's output, never written to a file
or committed) and deletes it from memory immediately after use.

**On a disconnect and re-run**: this clone lives on local Colab disk,
so it does NOT survive the disconnect (same as the code clone) -- but
everything pushed to this notebook's own `CKPT_BRANCH` before the
disconnect DID survive, on GitHub. Re-running this cell re-clones
fresh and fetches that branch if it already exists, so the run cell's
`resume_from` still finds the latest checkpoint and continues from
there, not round 0.


In [ ]:
# 7.5. Git-backed checkpoint durability (replaces Drive -- see markdown above).
import subprocess
from getpass import getpass

CKPT_REPO_DIR = "/content/FL-ZKP-checkpoints"
# 2026-11-17: each of the 4 MVP run notebooks gets its OWN branch, not
# one shared branch -- see FIX.md F17. Running all four at once on one
# shared branch caused every single push to collide and fail (12/12
# rounds, observed live) even with F16's fetch+rebase retry, because
# the retry itself can lose the SAME race under real concurrent load.
# A dedicated ref per notebook removes the race at the git level
# instead of mitigating it after the fact.
CKPT_BRANCH = "colab-checkpoints-ablation-case1-tiny_cnn"

def _git(args, cwd=CKPT_REPO_DIR, check=True):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed:\n{r.stdout}\n{r.stderr}")
    return r

if not os.path.exists(CKPT_REPO_DIR):
    _token = getpass("Paste a GitHub PAT with push access to YashasweeNarayan/FL-ZKP (hidden input, not stored): ").strip()
    _remote = f"https://{_token}@github.com/YashasweeNarayan/FL-ZKP.git"
    _git(["clone", "--depth", "1", _remote, CKPT_REPO_DIR], cwd=None)
    _git(["config", "user.email", "yashasweenarayan.yn@gmail.com"])
    _git(["config", "user.name", "Yashaswee Narayan"])
    # does this notebook's own CKPT_BRANCH already exist on the remote
    # (pushed in an earlier, since-disconnected session)? if so, use it --
    # that's where the actual latest checkpoint lives. If not, this is new.
    # explicit destination refspec -- a plain "fetch origin <branch>"
    # only updates FETCH_HEAD, not refs/remotes/origin/<branch>, so the
    # checkout below would fail even though the fetch itself succeeded.
    # Verified against a real throwaway git remote before using this.
    _fetch = _git(["fetch", "origin", f"{CKPT_BRANCH}:refs/remotes/origin/{CKPT_BRANCH}"], check=False)
    if _fetch.returncode == 0:
        _git(["checkout", "-B", CKPT_BRANCH, f"origin/{CKPT_BRANCH}"])
        print(f"found an existing '{CKPT_BRANCH}' branch on the remote -- resuming from it.")
    else:
        _git(["checkout", "-b", CKPT_BRANCH])
        print(f"no existing '{CKPT_BRANCH}' branch -- starting fresh.")
    # keep the remote URL with the embedded token only as long as needed
    # for this session's pushes; drop the plaintext token variable now.
    del _token
else:
    print("checkpoint checkout already present this session -- reusing it.")

CKPT_DIR = f"{CKPT_REPO_DIR}/_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def push_checkpoint_to_git(ckpt_path, round_idx):
    """on_checkpoint_saved callback for run_training_loop (fl/round_loop.py,
    2026-11-16). Never raises -- a push failure costs a retry next round,
    not the training run; the checkpoint is already safe on local disk
    regardless of whether this push succeeds.

    2026-11-17 (FIX.md F17): each MVP run notebook now has its OWN
    dedicated CKPT_BRANCH, so this retry is defense-in-depth, not the
    primary fix -- it used to be the ONLY mitigation for 4 notebooks
    sharing one branch, which collided on every single push under real
    concurrent load (observed live). Kept in case this exact branch is
    ever reused by two sessions at once (e.g. the same notebook run
    twice): fetch+rebase onto the remote tip and retry once before
    giving up for this round (next round tries again regardless)."""
    try:
        _git(["add", "-A"])
        commit = _git(["commit", "-m", f"checkpoint: round {round_idx}"], check=False)
        if commit.returncode != 0 and "nothing to commit" not in commit.stdout:
            print(f"  [checkpoint push WARNING] round {round_idx} commit failed: {commit.stdout.strip()}")
            return
        push = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
        if push.returncode != 0:
            # Lost a race with a concurrent writer on this branch --
            # fetch + rebase onto the remote tip and retry once.
            _git(["fetch", "origin", CKPT_BRANCH], check=False)
            rebase = _git(["rebase", f"origin/{CKPT_BRANCH}"], check=False)
            if rebase.returncode == 0:
                push2 = _git(["push", "-u", "origin", CKPT_BRANCH], check=False)
                if push2.returncode == 0:
                    print(f"  [checkpoint] round {round_idx}: lost a race, rebased, retried -- now pushed.")
                    return
            else:
                _git(["rebase", "--abort"], check=False)
            print(f"  [checkpoint push WARNING] round {round_idx} saved locally but push FAILED even after rebase retry "
                  f"(will retry next round): {push.stderr.strip()[-300:]}")
        else:
            print(f"  [checkpoint] round {round_idx} pushed to '{CKPT_BRANCH}' -- safe against a disconnect now.")
    except Exception as e:
        print(f"  [checkpoint push WARNING] round {round_idx}: {e!r} -- local checkpoint is still fine, will retry next round.")


In [ ]:
# 8. The ablation run: FORCE full participation (p=1.0), 2 rounds only.
#
# This is the ONLY cell that differs from the main
# RUN_MVP_TINY_CNN_MNIST.ipynb notebook. Everything
# above and below this cell (clone/install/sanity/config/dataset/ring/
# model/checkpoint-setup/eval) is the exact same code, imported
# verbatim from notebooks/_build_mvp_run_notebooks.py -- see this
# notebook's own intro markdown and ARCHITECTURE.md SS5.5.1.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint

# FIX.md F21 (2026-11-17): the ablation's whole point -- override the
# shipped config's sampling_rate (0.25) to 1.0 so the VRF self-sampling
# gate (fl/client.py's vrf_is_sampled, ARCHITECTURE.md SS5.5) passes for
# EVERY one of the 50 clients. Zero dummy (q_k=0) submissions this run;
# every accepted submission is a real, locally-trained delta. This is a
# config mutation in the notebook's own CFG dict, not a library change --
# run_training_loop reads CFG['protocol']['sampling_rate'] fresh every
# call (fl/round_loop.py), so nothing in fl/ needs to know this is an
# ablation.
CFG['protocol']['sampling_rate'] = 1.0
ABLATION_SAMPLING_RATE = CFG['protocol']['sampling_rate']
REAL_ROUNDS = 2  # deliberately NOT 20 -- see intro markdown for why.

# '_ablation_p1_r2' suffix: distinct from every main-matrix and
# attack-matrix checkpoint filename by construction, so this can never
# be mistaken for, or accidentally resumed as, a real training run.
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_ablation_p1_r{REAL_ROUNDS}.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"ablation-bootstrap-{REAL_MODEL}-case1".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found ablation checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    on_checkpoint_saved=push_checkpoint_to_git,
)
release_gpu_memory()

print("="*70)
print(f"ABLATION (p={ABLATION_SAMPLING_RATE}) RESULT -- {REAL_MODEL} + MNIST")
print("="*70)
for h in real_history:
    if h.get('aborted'):
        print(f"  round {h.get('round')}: ABORTED -- {h.get('reason')} <-- UNEXPECTED at p=1.0, report this")
    else:
        n_acc = h.get('accepted')
        flag = "OK (all 50 real, no dummies)" if n_acc == n_clients else f"UNEXPECTED: only {n_acc}/{n_clients} accepted at p=1.0"
        print(f"  round {h.get('round')}: accepted={n_acc}/{n_clients}, aggregate_exact={h.get('aggregate_exact')} -- {flag}")

n_ok_rounds = sum(1 for h in real_history if not h.get('aborted') and h.get('accepted') == n_clients)
if n_ok_rounds == REAL_ROUNDS:
    print(f"\nPASS: all {REAL_ROUNDS} rounds completed with all {n_clients} clients accepted as real contributors -- "
          f"the protocol is NOT structurally limited to p<1.0, subsampling is a config choice (ARCHITECTURE.md SS5.5.1).")
else:
    print(f"\nINCONCLUSIVE/FAIL: only {n_ok_rounds}/{REAL_ROUNDS} rounds had all {n_clients} clients accepted -- investigate before citing this as the full-participation proof.")


## Evaluate accuracy

Works whether you let training finish, stopped it early (Ctrl-C /
interrupt), or are coming back in a fresh runtime with nothing but the
checkpoint pushed to this notebook's own `CKPT_BRANCH`: it loads the LATEST checkpointed weights directly
from `CKPT_PATH` (falling back to the in-memory `final_w_t` only if no
checkpoint file exists yet), so it always reports accuracy for
whatever round you actually reached, not a partially-updated `model`
object.


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    # GPU fix (2026-11-16): dataset tensors stay on CPU by design (see
    # fl/client.py's 2026-11-16 comment); move each eval batch to
    # wherever `model` actually lives, same pattern as training.
    device = next(model.parameters()).device
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            batch_images = images[i:i + batch_size].to(device)
            batch_labels = labels[i:i + batch_size].to(device)
            logits = model(batch_images)
            preds = logits.argmax(dim=1)
            correct += (preds == batch_labels).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- see markdown above.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t.to(next(model.parameters()).device), model_shapes))

# Official MNIST test set (10,000 images) -- the headline number.
test_images, test_labels = case1_data['test']
test_acc = evaluate_accuracy(model, test_images, test_labels)

# global_val: a held-out slice of the TRAINING set (disjoint from every
# client's own data, see fl/splits.py) -- a secondary sanity check, not
# the headline number, since it's drawn from the same distribution the
# clients trained on rather than the official test split.
val_ds = case1_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: official MNIST test accuracy = {test_acc:.4f} ({len(test_images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")
